Cleaning cohort retention analysis

In [1]:
import pandas as pd

cohort_retention_analysis = pd.read_csv(r"C:\Users\chase\OneDrive\Desktop\Adventure Works Project\Raw csvs\cohort_retention_analysis.csv")
cohort_retention_analysis.info()
cohort_retention_analysis.head()

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   cohort_quarter                 13 non-null     str    
 1   cohort_quarter_customer_count  13 non-null     int64  
 2   quarter_1_customer_count       12 non-null     float64
 3   quarter_1_retention            12 non-null     float64
 4   quarter_2_customer_count       11 non-null     float64
 5   quarter_2_retention            11 non-null     float64
 6   quarter_3_customer_count       10 non-null     float64
 7   quarter_3_retention            10 non-null     float64
dtypes: float64(6), int64(1), str(1)
memory usage: 964.0 bytes


,cohort_quarter,cohort_quarter_customer_count,quarter_1_customer_count,quarter_1_retention,quarter_2_customer_count,quarter_2_retention,quarter_3_customer_count,quarter_3_retention
0,2022-04-01,264,109.0,41.29,106.0,40.15,107.0,40.53
1,2022-07-01,545,65.0,11.93,71.0,13.03,69.0,12.66
2,2022-10-01,603,7.0,1.16,10.0,1.66,7.0,1.16
3,2023-01-01,589,1.0,0.17,0.0,0.00,0.0,0.00
4,2023-04-01,798,116.0,14.54,107.0,13.41,113.0,14.16


Must change "cohort_quarter from str to datetime.

In [2]:
cohort_retention_analysis["cohort_quarter"] = pd.to_datetime(cohort_retention_analysis["cohort_quarter"])

cohort_retention_analysis.info()

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   cohort_quarter                 13 non-null     datetime64[us]
 1   cohort_quarter_customer_count  13 non-null     int64         
 2   quarter_1_customer_count       12 non-null     float64       
 3   quarter_1_retention            12 non-null     float64       
 4   quarter_2_customer_count       11 non-null     float64       
 5   quarter_2_retention            11 non-null     float64       
 6   quarter_3_customer_count       10 non-null     float64       
 7   quarter_3_retention            10 non-null     float64       
dtypes: datetime64[us](1), float64(6), int64(1)
memory usage: 964.0 bytes


Renamed the columns to melt using .wide_to_long, which requires the number at the end of the column name.

In [3]:
cohort_retention_analysis = cohort_retention_analysis.rename(columns = {
    "quarter_1_customer_count" : "customer_count_1",
    "quarter_1_retention" : "retention_1",
    "quarter_2_customer_count" : "customer_count_2",
    "quarter_2_retention" : "retention_2",
    "quarter_3_customer_count" : "customer_count_3",
    "quarter_3_retention" : "retention_3"
})

cohort_retention_analysis.columns


Index(['cohort_quarter', 'cohort_quarter_customer_count', 'customer_count_1',
       'retention_1', 'customer_count_2', 'retention_2', 'customer_count_3',
       'retention_3'],
      dtype='str')

Melted to long format so that customer_count and retention for each quarter since cohort can be a dimension instead of separate measures in Tableau.

In [4]:
cohort_retention_long = pd.wide_to_long(
    cohort_retention_analysis,
    stubnames = ["customer_count", "retention"],
    i = ["cohort_quarter", "cohort_quarter_customer_count"],
    j = "quarters_since_cohort",
    sep = "_"
).reset_index()

cohort_retention_long.head(10)

,cohort_quarter,cohort_quarter_customer_count,quarters_since_cohort,customer_count,retention
0,2022-04-01,264,1,109.0,41.29
1,2022-04-01,264,2,106.0,40.15
2,2022-04-01,264,3,107.0,40.53
3,2022-07-01,545,1,65.0,11.93
4,2022-07-01,545,2,71.0,13.03
5,2022-07-01,545,3,69.0,12.66
6,2022-10-01,603,1,7.0,1.16
7,2022-10-01,603,2,10.0,1.66
8,2022-10-01,603,3,7.0,1.16
9,2023-01-01,589,1,1.0,0.17


Creating this "is_measurable" boolean will allow me to decide how I want to handle the null values in Tableau for visualizations. The nulls represent customer counts that couldn't be determined towards the end of the data. Double checked that the null values for customer count and retention lined up together.

In [10]:
cohort_retention_long["is_measurable"] = cohort_retention_long["customer_count"].notna()

cohort_retention_long["is_measurable"].value_counts()

(cohort_retention_long["customer_count"].isna() == cohort_retention_long["retention"].isna()).all()

np.True_

In [11]:
cohort_retention_long.to_csv(r"C:\Users\chase\OneDrive\Desktop\Adventure Works Project\Clean csvs\cohort_retention_analysis_cleaned.csv", index = False)